In [1]:
# notebooks/03_preprocessing.ipynb — CELL 1

import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder

import joblib
import os

# -----------------------------------
# CREATE MODELS FOLDER
# -----------------------------------
os.makedirs('../models', exist_ok=True)

# -----------------------------------
# LOAD DATASET
# -----------------------------------
df = pd.read_csv('../data/fd_loans.csv')

# -----------------------------------
# LABEL ENCODING
# -----------------------------------
le_emp = LabelEncoder()
le_city = LabelEncoder()
le_bank = LabelEncoder()

df['emp_enc'] = le_emp.fit_transform(df['employment'])

df['city_enc'] = le_city.fit_transform(df['city_tier'])

df['bank_enc'] = le_bank.fit_transform(df['fd_bank'])

# -----------------------------------
# FEATURE ENGINEERING
# -----------------------------------

# 1. FD Coverage Ratio
df['fd_coverage'] = (
    df['fd_amount'] / (df['loan_amt'] + 1)
)

# 2. FD Maturity Buffer
df['maturity_buffer'] = (
    df['fd_months_to_maturity']
    - df['loan_tenure_mo']
)

# 3. Interest Spread
df['interest_spread'] = (
    df['loan_interest']
    - df['fd_interest_rate']
)

# 4. LTV Band Number
df['ltv_band_num'] = pd.cut(
    df['ltv'],
    bins=[0, 0.60, 0.70, 0.80, 0.90, 1.0],
    labels=[0, 1, 2, 3, 4]
).astype(int)

# 5. High LTV Flag
df['high_ltv'] = (
    df['ltv'] > 0.80
).astype(int)

# 6. Relationship Score
df['relationship_score'] = (
    df['relationship_yrs']
    * df['digital_user']
    * (1 - df['pre_closure_risk'])
)

# 7. Debt Burden
df['debt_burden'] = (
    df['num_existing_loans']
    * df['emi_income']
)

# 8. FD Density
df['fd_density'] = (
    df['fd_amount']
    / (df['fd_tenure_mo'] / 12 + 1)
)

# -----------------------------------
# PRINT ENGINEERED FEATURES
# -----------------------------------
print("8 ENGINEERED FEATURES:\n")

feats = [
    'fd_coverage',
    'maturity_buffer',
    'interest_spread',
    'ltv_band_num',
    'high_ltv',
    'relationship_score',
    'debt_burden',
    'fd_density'
]

for f in feats:
    print(f"{f:22}: mean = {df[f].mean():.4f}")

# -----------------------------------
# SAVE LABEL ENCODERS
# -----------------------------------
joblib.dump(le_emp, '../models/le_emp.pkl')

joblib.dump(le_city, '../models/le_city.pkl')

joblib.dump(le_bank, '../models/le_bank.pkl')

# -----------------------------------
# SAVE ENGINEERED DATASET
# -----------------------------------
df.to_csv(
    '../data/fd_loans_engineered.csv',
    index=False
)

print("\nSaved: data/fd_loans_engineered.csv")

print("Saved encoders:")
print("- le_emp.pkl")
print("- le_city.pkl")
print("- le_bank.pkl")

8 ENGINEERED FEATURES:

fd_coverage           : mean = 1.5721
maturity_buffer       : mean = 4.7208
interest_spread       : mean = 1.6553
ltv_band_num          : mean = 1.4503
high_ltv              : mean = 0.2719
relationship_score    : mean = 7.1929
debt_burden           : mean = 1.0132
fd_density            : mean = 101619.9383

Saved: data/fd_loans_engineered.csv
Saved encoders:
- le_emp.pkl
- le_city.pkl
- le_bank.pkl
